# 02 — Geometric Features from a Dimer xyz File

This notebook walks through the **11-dimensional symmetry-adapted feature vector**
that is the geometric input to the ML model. We start from a single dimer xyz file
and produce one feature row.

## What problem the features solve

The reference coupling $J$ is invariant under any global rotation/translation of
the dimer, and (because each Fe-porphine has $D_{4h}$ symmetry in isolation) under
$C_4$ rotations about the porphyrin normal of either monomer. A naive Cartesian
representation has no such invariance, so the model would have to learn it from
data (a lot of data). We instead encode invariance into the features:

1. Compute a local frame on each monomer from its 4 porphyrin-core nitrogens.
2. Express the relative position of monomer 2 in monomer 1's frame.
3. Use $\cos(4\theta)$ and $\sin(4\theta)$ for any in-plane angle, so $C_4$
   rotations leave the features unchanged.


## Setup

In [1]:
import sys, os, numpy as np
sys.path.insert(0, '../src')
from geometry import load_dimer_xyz, monomer_frame, dimer_features


## Step 1: Load a dimer xyz

A dimer xyz file is a standard XYZ format with the two monomers' atoms listed
sequentially: first the 37 atoms of monomer 1, then the 37 atoms of monomer 2.

We use the convenience function `load_dimer_xyz` which returns a dict with
`monomer1` and `monomer2` keys.

In [2]:
# Use one example from the systematic 704 set
xyz_path = '../data/structures/35_ang_30_30.xyz'  # 3.5 A separation, 30° lateral slip
dimer = load_dimer_xyz(xyz_path)

print(f"Total atoms: {len(dimer['symbols'])}")
print(f"Monomer 1: {len(dimer['monomer1']['symbols'])} atoms, "
      f"composition {dict(zip(*np.unique(dimer['monomer1']['symbols'], return_counts=True)))}")
print(f"Monomer 2: {len(dimer['monomer2']['symbols'])} atoms")


Total atoms: 74
Monomer 1: 37 atoms, composition {np.str_('C'): np.int64(20), np.str_('Fe'): np.int64(1), np.str_('H'): np.int64(12), np.str_('N'): np.int64(4)}
Monomer 2: 37 atoms


## Step 2: Compute the local frame of each monomer

`monomer_frame(positions, symbols)` returns:

- **centroid**: mean position of the 4 porphyrin-core nitrogens
- **normal**: unit vector perpendicular to the porphyrin plane (sign chosen later)
- **x_axis**: in-plane vector pointing toward the first nitrogen (in array order)

The four nitrogens are identified as the four N atoms closest to the Fe
center — this is robust to peptoid linkers or amide nitrogens elsewhere
in the molecule (relevant for MD-sampled Fe-TPP).

In [3]:
m1 = dimer['monomer1']
m2 = dimer['monomer2']

c1, n1, x1 = monomer_frame(m1['positions'], m1['symbols'])
c2, n2, x2 = monomer_frame(m2['positions'], m2['symbols'])

print(f"Monomer 1 frame:")
print(f"  centroid = {c1.round(3)}")
print(f"  normal   = {n1.round(3)}")
print(f"  x-axis   = {x1.round(3)}")
print()
print(f"Monomer 2 frame:")
print(f"  centroid = {c2.round(3)}")
print(f"  normal   = {n2.round(3)}")
print(f"  x-axis   = {x2.round(3)}")


Monomer 1 frame:
  centroid = [ 0.001 -0.006  0.218]
  normal   = [-0.  0.  1.]
  x-axis   = [1.    0.019 0.   ]

Monomer 2 frame:
  centroid = [3.001 2.994 3.718]
  normal   = [-0.  0.  1.]
  x-axis   = [1.    0.019 0.   ]


## Step 3: Compute the 11-dimensional feature vector

`dimer_features(...)` returns 13 features in total; the first 11 are what the v2
geometry model uses. The last 2 are an Hückel-like overlap surrogate retained for
backward compatibility.

The 11 features:

| Index | Name | Meaning |
|---|---|---|
| 0 | `dz` | interplanar distance (along $\mathbf{n}_1$), Å, always $> 0$ |
| 1 | `dx` | lateral slip along $\mathbf{x}_1$, Å |
| 2 | `dy` | lateral slip along $\mathbf{y}_1 = \mathbf{n}_1 \times \mathbf{x}_1$, Å |
| 3 | `cos_theta` | $\mathbf{n}_1 \cdot \mathbf{n}_2$ — face-to-face = +1, edge-on = 0 |
| 4-5 | `cos_4phi`, `sin_4phi` | in-plane twist $\varphi$ between $\mathbf{x}_1, \mathbf{x}_2$ projected on monomer-1 plane, mapped to $4\varphi$ for $C_4$ invariance |
| 6-7 | `cos_4psi`, `sin_4psi` | direction of the tilt axis $\mathbf{n}_1 \times \mathbf{n}_2$ relative to $\mathbf{x}_1$ |
| 8 | `d_lateral` | $\sqrt{dx^2 + dy^2}$, the unsigned slip magnitude |
| 9-10 | `cos_4alpha`, `sin_4alpha` | direction of slip $\arctan(dy/dx)$, mapped to $4\alpha$ |

In [4]:
features = dimer_features(m1['positions'], m1['symbols'],
                          m2['positions'], m2['symbols'])

names = ['dz', 'dx', 'dy', 'cos_theta',
         'cos_4phi', 'sin_4phi', 'cos_4psi', 'sin_4psi',
         'd_lateral', 'cos_4alpha', 'sin_4alpha',
         '|S_HL|', 'S_HL^2']

print(f"{'idx':>3} {'name':<12} value")
for i, (n, v) in enumerate(zip(names, features)):
    print(f"{i:>3} {n:<12} {v:+.4f}")


idx name         value
  0 dz           +3.4999
  1 dx           +3.0576
  2 dy           +2.9414
  3 cos_theta    +1.0000
  4 cos_4phi     +1.0000
  5 sin_4phi     -0.0000
  6 cos_4psi     +1.0000
  7 sin_4psi     +0.0000
  8 d_lateral    +4.2427
  9 cos_4alpha   -0.9970
 10 sin_4alpha   +0.0774
 11 |S_HL|       +0.0006
 12 S_HL^2       +0.0000


## Step 4: Verify $D_{4h}$ invariance

If we rotate monomer 2 by 90° around its own normal axis, the porphyrin is
mapped to itself (it has $C_4$ symmetry). The features 4-7 and 9-10 should be
unchanged because of the $4\theta$ wrapping. Let's check.

In [5]:
from scipy.spatial.transform import Rotation

# Rotate monomer 2 by 90° around its own normal axis (about its centroid)
R = Rotation.from_rotvec(0.5 * np.pi * n2).as_matrix()
m2_rotated_pos = (m2['positions'] - c2) @ R.T + c2

features_rotated = dimer_features(m1['positions'], m1['symbols'],
                                  m2_rotated_pos, m2['symbols'])

print(f"{'name':<12} {'original':>12} {'after 90° rot':>16} {'diff':>10}")
for i, n in enumerate(names):
    diff = features_rotated[i] - features[i]
    flag = '  <-- changed!' if abs(diff) > 1e-6 else ''
    print(f"{n:<12} {features[i]:>12.4f} {features_rotated[i]:>16.4f} {diff:>+10.4e}{flag}")


name             original    after 90° rot       diff
dz                 3.4999           3.4999 +0.0000e+00
dx                 3.0576           3.0576 +0.0000e+00
dy                 2.9414           2.9414 +0.0000e+00
cos_theta          1.0000           1.0000 +0.0000e+00
cos_4phi           1.0000           1.0000 +0.0000e+00
sin_4phi          -0.0000          -0.0000 -9.0070e-18
cos_4psi           1.0000           1.0000 +0.0000e+00
sin_4psi           0.0000           0.0000 +0.0000e+00
d_lateral          4.2427           4.2427 +0.0000e+00
cos_4alpha        -0.9970          -0.9970 +0.0000e+00
sin_4alpha         0.0774           0.0774 +0.0000e+00
|S_HL|             0.0006           0.0006 -1.2480e-07
S_HL^2             0.0000           0.0000 -1.6047e-10


All 11 features (indices 0-10) are unchanged by the 90° rotation. Only the
Hückel-like features 11-12 change (slightly), because they depend on the actual
positions of the alpha-C atoms which DO get permuted by the 90° rotation.

## Step 5: Process the 704 systematic dataset

Now we use the same pipeline to compute features for all 704 structures and
verify they match what's in the master CSV.

In [6]:
import glob, pandas as pd

xyz_files = sorted(glob.glob('../data/structures/*.xyz'))[:5]  # just first 5 for speed
df_master = pd.read_csv('../data/master_dataset.csv')

print(f"{'name':<22} {'dz':>7} {'d_lat':>7} {'cos_th':>7} {'matches CSV':>12}")
for fn in xyz_files:
    name = os.path.splitext(os.path.basename(fn))[0]
    d = load_dimer_xyz(fn)
    f = dimer_features(d['monomer1']['positions'], d['monomer1']['symbols'],
                       d['monomer2']['positions'], d['monomer2']['symbols'])
    csv_row = df_master[df_master['name'] == name].iloc[0]
    csv_dz = csv_row['dz']; csv_dlat = csv_row['d_lateral']; csv_cos = csv_row['cos_theta']
    match = (abs(f[0]-csv_dz)<1e-4 and abs(f[8]-csv_dlat)<1e-4 and abs(f[3]-csv_cos)<1e-4)
    print(f"{name:<22} {f[0]:>7.3f} {f[8]:>7.3f} {f[3]:>+7.3f}  {'OK' if match else 'MISMATCH':>12}")


name                        dz   d_lat  cos_th  matches CSV
35_ang_02                3.500   0.200  +1.000            OK
35_ang_02_02             3.500   0.283  +1.000            OK
35_ang_04                3.500   0.400  +1.000            OK
35_ang_04_04             3.500   0.566  +1.000            OK
35_ang_06                3.500   0.600  +1.000            OK


## Summary

- Reading a dimer xyz and computing the 11-D feature vector is a two-line operation.
- The features are invariant under translation, global rotation, AND $C_4$ rotation
  of either monomer about its own porphyrin normal.
- This invariance is built into the features by construction, not learned from data.

In notebook 03, we add the $S_{12}$ orbital overlap features that turn the v2
geometry model into the recommended v4_srsh_aug model.
